# COMANDOS AVANZADOS EN SPARK
#### Se utilizaran estos comandos para la particion de datos, exportacion de informacion y finalmente ejemplos simples de python conectado con Spark

---------------------

### Conexión con Spark

In [10]:
#Se importan librerias a utilizar
from pyspark.sql import SparkSession 
from pyarrow import csv 
import pyspark.sql.functions as F1
import pyarrow as pa 
import pyarrow.parquet as pq 
import pandas as pd 

In [3]:
#Se crea cliente de spark para python
spark = SparkSession \
    .builder \
    .appName('LecturaArchivocsv') \
    .getOrCreate()

In [11]:
#Se realiza un chequeo del archivo importado
path_archivo = 'databses/vgsales.csv'
vg = spark.read.csv(path_archivo, header = True)

vg.show()

+----+--------------------+--------+----+------------+--------------------+--------+--------+--------+-----------+------------+
|Rank|                Name|Platform|Year|       Genre|           Publisher|NA_Sales|EU_Sales|JP_Sales|Other_Sales|Global_Sales|
+----+--------------------+--------+----+------------+--------------------+--------+--------+--------+-----------+------------+
|   1|          Wii Sports|     Wii|2006|      Sports|            Nintendo|   41.49|   29.02|    3.77|       8.46|       82.74|
|   2|   Super Mario Bros.|     NES|1985|    Platform|            Nintendo|   29.08|    3.58|    6.81|       0.77|       40.24|
|   3|      Mario Kart Wii|     Wii|2008|      Racing|            Nintendo|   15.85|   12.88|    3.79|       3.31|       35.82|
|   4|   Wii Sports Resort|     Wii|2009|      Sports|            Nintendo|   15.75|   11.01|    3.28|       2.96|          33|
|   5|Pokemon Red/Pokem...|      GB|1996|Role-Playing|            Nintendo|   11.27|    8.89|   10.22|  

In [12]:
#Se realiza una consulta sencilla
vg.sort(F1.col('Platform')).show(10)

+----+---------------+--------+----+--------+------------+--------+--------+--------+-----------+------------+
|Rank|           Name|Platform|Year|   Genre|   Publisher|NA_Sales|EU_Sales|JP_Sales|Other_Sales|Global_Sales|
+----+---------------+--------+----+--------+------------+--------+--------+--------+-----------+------------+
| 545|Missile Command|    2600|1980| Shooter|       Atari|    2.56|    0.17|       0|       0.03|        2.76|
|1431|      Centipede|    2600|1981| Shooter|       Atari|    1.26|    0.08|       0|       0.01|        1.36|
| 608| Space Invaders|    2600| N/A| Shooter|       Atari|    2.36|    0.14|       0|       0.03|        2.53|
| 240|       Pitfall!|    2600|1981|Platform|  Activision|    4.21|    0.24|       0|       0.05|         4.5|
| 736|        Frogger|    2600|1981|  Action|Parker Bros.|    2.06|    0.12|       0|       0.02|         2.2|
|1108|    Ms. Pac-Man|    2600|1981|  Puzzle|       Atari|    1.54|     0.1|       0|       0.02|        1.65|
|

<b>INSIGHTS</b>
<p>Lo que se obtuvo dentro de esta consulta basica fueron los primeros diez datos que puedan para una lectura y visualizacion resumida y sencilla del dataset presente, que aun que no haya sido especificado en la instruccion se utilizara este como ejemplo de los siguientes puntos</p>

--------------------

### Particion de datos

#### Partition by()

In [13]:
#Implementacion de PArititionBy
spark = SparkSession.builder.appName('Partitionby() PySpark').getOrCreate()

#Leer el dataframe siendo el archivo de vgsales
dataframe = spark.read.option('header',True).csv('databses/vgsales.csv')

#Imprimir el esquema del archivo
dataframe.printSchema()


root
 |-- Rank: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Platform: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Genre: string (nullable = true)
 |-- Publisher: string (nullable = true)
 |-- NA_Sales: string (nullable = true)
 |-- EU_Sales: string (nullable = true)
 |-- JP_Sales: string (nullable = true)
 |-- Other_Sales: string (nullable = true)
 |-- Global_Sales: string (nullable = true)



In [ ]:
#Usando partitionby mas la columna de Genre
dataframe.write.option('header',True).partitionBy('Genre').mode('overwrite').csv('databses/vgsales')

In [ ]:
#Particionando pero solo hasta cierto limite numero de datos
dataframe.write.option('header',True).option('maxRecordsPerFile',10).partitionBy('Year').mode('overwrite').csv('databses/vgsales/years')

#### Repartition by()

In [17]:
#Implementacion de PArititionBy
spark = SparkSession.builder.appName('coalesce() PySpark').getOrCreate()

#Leer el dataframe siendo el archivo de vgsales
dataco = spark.read.option('header',True).csv('databses/vgsales.csv')

#Imprimir el esquema del archivo
dataco.printSchema()

root
 |-- Rank: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Platform: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Genre: string (nullable = true)
 |-- Publisher: string (nullable = true)
 |-- NA_Sales: string (nullable = true)
 |-- EU_Sales: string (nullable = true)
 |-- JP_Sales: string (nullable = true)
 |-- Other_Sales: string (nullable = true)
 |-- Global_Sales: string (nullable = true)



In [ ]:
#Repartition
#Generar el directorio el numeo de archivos señalados(20)
dataco.repartition(20).write.mode('overwrite').option('header', True).csv('databses/vgsales/vg')

In [ ]:
#Como se explico se puede cambiar el numero de reparticiones y lo hara inmediatamente
dataco.repartition(30).write.mode('overwrite').option('header',True).csv('databses/vgsales/vg')

In [ ]:
#nuevo ejemplo con 10 particiones, el directorio se genera automaticamente
dataco.repartition(10).write.mode('overwrite').option('header', True).csv('databses/vgsales/ej10')

#### Coalesce

In [21]:
#Usando Coalesce
#A diferencia de raepartition, que ejecuta un shuffle para rearmar el # de particiones, coalesce sirve para disminuir el numero de las mismas 
#Coalesce no aplica un full reshuffle a menos que sea necesario
#menos pesado computacionalmente

#Se obtiene el numero de particiones por defecto
dataco_2 = dataco.repartition(20)
dataco_2.rdd.getNumPartitions()

20

In [ ]:
#Se interpreta la siguiente variable 
dataco_3 = dataco_2.coalesce(10)

In [23]:
dataco_3.rdd.getNumPartitions()

10

In [ ]:
#Se genera el directorio donde reducira el numero de particiones del DAtaframe
dataco_3.write.mode('overwrite').option('header',True).csv('databses/vgsales/coalesce')

<b>INSIGHTS</b>
<p>Lo que se pudo encontrar dentro de estos comandos es que al momento de realizar varios se pueden eliminar los resultados anteriores por lo que es importante que se realicen las consultas necesarias antes de que se llame otro comando y se remplace la información previa</p>

-----------------------------

##  USO DE FORMATOS DE CREACION DE TABLAS A PANDAS

### PyArrow

In [26]:
#Llamar el archivo con el que se ha estado trabajando
archivo = 'databses/vgsales.csv'
tab_tables =csv.read_csv(archivo)

In [27]:
#Estructura de vgsales visto desde PyArrow
tab_tables

pyarrow.Table
Rank: int64
Name: string
Platform: string
Year: int64
Genre: string
Publisher: string
NA_Sales: double
EU_Sales: double
JP_Sales: double
Other_Sales: double
Global_Sales: double
----
Rank: [[1,2,3,4,5,...,12661,12662,12663,12664,12665],[12666,12667,12668,12669,12670,...,16596,16597,16598,16599,16600]]
Name: [["Wii Sports","Super Mario Bros.","Mario Kart Wii","Wii Sports Resort","Pokemon Red/Pokemon Blue",...,"Hajime no Ippo Portable: Victorious Spirits","Nanostray","CMT Presents: Karaoke Revolution Country","Katekyoo Hitman Reborn! DS: Flame Rumble Mukuro Kyoushuu","Cid to Chocobo no Fushigi na Dungeon: Toki Wasure no Meikyuu DS+"],["Vitamin X to Z","Fatal Frame","Flip's Twisted World","Tom Clancy's Ghost Recon: Advanced Warfighter","Urban Chaos: Riot Response",...,"Woody Woodpecker in Crazy Castle 5","Men in Black II: Alien Escape","SCORE International Baja 1000: The Official Game","Know How 2","Spirits & Spells"]]
Platform: [["Wii","NES","Wii","Wii","GB",...,"PSP","DS",

In [28]:
#Saber el largo de la tabla generada a partir de un
len(tab_tables)

16598

In [29]:
#Lista de columnas
tab_tables.column_names

['Rank',
 'Name',
 'Platform',
 'Year',
 'Genre',
 'Publisher',
 'NA_Sales',
 'EU_Sales',
 'JP_Sales',
 'Other_Sales',
 'Global_Sales']

In [30]:
#Chequeo del esquema
tab_tables.schema

Rank: int64
Name: string
Platform: string
Year: int64
Genre: string
Publisher: string
NA_Sales: double
EU_Sales: double
JP_Sales: double
Other_Sales: double
Global_Sales: double

In [31]:
#Uso de mas funciones para su descripcion
tab_tables.num_columns

11

In [32]:
tab_tables.num_rows

16598

In [34]:
#Se convierte esta estructura a pandas para una mejor visualizacion
pandas = tab_tables.to_pandas()

pandas.head()

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


In [35]:
#Agrupa la tabla por Genero 
Tab_genre = tab_tables.group_by('Genre').aggregate([('JP_Sales','sum')])

In [36]:
#Pasar resultado a pandas
topandas = Tab_genre.to_pandas()

In [41]:
topandas.head()

,Genre,JP_Sales_sum
0,Adventure,52.07
1,Action,159.95
2,Platform,130.77
3,Shooter,38.28
4,Role-Playing,352.31


### Formato Parquet

In [47]:
#Se genera un dataframe para transformacion a esquema
nuevodf = pd.DataFrame({'col1':[934,89324,54],
                        'col2':['oli','q','tal'],
                        'col3':[True,False,False]},
                        index = list('abc'))

In [48]:
#Chequeo del dataframe generado
nuevodf

,col1,col2,col3
a,934,oli,True
b,89324,q,False
c,54,tal,False


In [49]:
tab_silly = pa.Table.from_pandas(nuevodf)
pq.write_table(tab_silly,'databses/parquet/tab')

In [51]:
#Leyendo el archivo generado en formato parquet
tabtab = pq.read_table('databses/parquet/tab')
tabtab.to_pandas()

,col1,col2,col3
a,934,oli,True
b,89324,q,False
c,54,tal,False
